# Q-Learning on CliffWalking-v1

This notebook demonstrates **Q-Learning** (Watkins, 1989), an **off-policy Temporal Difference (TD) control** algorithm.

### Mathematical Update Rule
3633
Q(S_t, A_t) \leftarrow Q(S_t, A_t) + lpha \left[ R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a) - Q(S_t, A_t) ight]
3633

Because Q-learning updates values using $\max_a Q(S_{t+1}, a)$, it estimates the return of the **optimal policy** directly, irrespective of the exploratory actions taken by its $\epsilon569Xgreedy behavior policy.

In [ ]:
# Install Gymnasium if running in Colab/Fresh environment
# !pip install gymnasium pygame matplotlib numpy

import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt


### 1. Hyperparameters & Environment Setup

In [ ]:
gamma = 0.99     # Discount factor
epsilon = 0.1    # Exploration probability
alpha = 0.5      # Learning rate (step size)
episodes = 500   # Total training episodes

env = gym.make("CliffWalking-v1")
num_states = env.observation_space.n   # 48 states
num_actions = env.action_space.n       # 4 actions: UP(0), RIGHT(1), DOWN(2), LEFT(3)

# Initialize Q-table with zeros
Q = np.zeros((num_states, num_actions))


### 2. Epsilon-Greedy Policy

In [ ]:
def epsilon_greedy(state, eps=epsilon):
    """Select an action using epsilon-greedy exploration."""
    if random.random() < eps:
        return env.action_space.sample()  # Explore
    else:
        # Break ties randomly among best actions
        q_s = Q[state]
        best_actions = np.where(q_s == np.max(q_s))[0]
        return np.random.choice(best_actions)  # Exploit


### 3. Training Loop

In [ ]:
episode_rewards = []
episode_lengths = []

for episode in range(episodes):
    state, _ = env.reset()
    done = False
    total_reward = 0
    step_count = 0

    while not done:
        action = epsilon_greedy(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Q-Learning Bellman Optimality Update
        td_target = reward + (0 if done else gamma * np.max(Q[next_state]))
        Q[state, action] += alpha * (td_target - Q[state, action])

        state = next_state
        total_reward += reward
        step_count += 1

    episode_rewards.append(total_reward)
    episode_lengths.append(step_count)

    if (episode + 1) % 50 == 0:
        avg_r = np.mean(episode_rewards[-50:])
        print(f"Episode {episode+1:3d}/{episodes} | Average Reward: {avg_r:.1f}")

env.close()


### 4. Evaluation (What Did the Agent Learn?)

In [ ]:
# Evaluate purely greedy behavior
test_env = gym.make("CliffWalking-v1")
state, _ = test_env.reset()
done = False
total_reward = 0
path = [state]

while not done:
    action = np.argmax(Q[state])
    state, reward, terminated, truncated, _ = test_env.step(action)
    done = terminated or truncated
    path.append(state)
    total_reward += reward

print(f"Evaluation Total Reward: {total_reward}")
print(f"Evaluation Path Length: {len(path) - 1} steps")
print(f"Path Visited: {path}")
test_env.close()
